### Para evaluar nuestras propias fotos :)

Estas se encuentra en TestPropio/[nombre]/[letra].jpg

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tensorflow.keras.models import load_model
import joblib

TAM_OBJETIVO = (64, 64)

modelo = load_model("mejor_modelo_asl.keras")
codificador = joblib.load("codificador_asl.pkl")

print("Modelo y encoder cargados correctamente :D.")

Modelo y encoder cargados correctamente :D.


In [8]:
def predecir_imagen_propia(ruta_imagen, modelo, codificador, tam=TAM_OBJETIVO, mostrar=True):
    img_original = Image.open(ruta_imagen).convert("RGB")
    img_resized = img_original.resize(tam)
    arr = np.array(img_resized, dtype=np.float32) / 255.0
    arr = np.expand_dims(arr, axis=0)

    pred_prob = modelo.predict(arr, verbose=0)[0]
    idx_pred = np.argmax(pred_prob)
    clase_pred = codificador.inverse_transform([idx_pred])[0]
    confianza = pred_prob[idx_pred]

    if mostrar:
        plt.figure(figsize=(3, 3))
        plt.imshow(img_original)
        plt.title(f"Predicción: {clase_pred} ({confianza:.1%})")
        plt.axis("off")
        plt.show()

    return clase_pred, confianza


def predecir_lote(rutas_imagenes, etiquetas_reales, modelo, codificador, tam=TAM_OBJETIVO, integrante=None):
    resultados = []
    for ruta, real in zip(rutas_imagenes, etiquetas_reales):
        pred, conf = predecir_imagen_propia(ruta, modelo, codificador, tam, mostrar=False)
        resultados.append({
            "integrante": integrante,
            "archivo": ruta,
            "real": real,
            "predicho": pred,
            "confianza": round(float(conf), 4),
            "correcto": pred == real
        })

    df_resultados = pd.DataFrame(resultados)
    aciertos = df_resultados["correcto"].sum()
    total = len(df_resultados)
    titulo = f" ({integrante})" if integrante else ""
    print(f"Aciertos{titulo}: {aciertos}/{total} ({aciertos/total:.1%})")
    return df_resultados

In [ ]:
rutas_melisa = ["./TestPropio/Melisa/M.jpg", "./TestPropio/Melisa/E.jpg", "./TestPropio/Melisa/L.jpg",
                "./TestPropio/Melisa/I.jpg", "./TestPropio/Melisa/S.jpg", "./TestPropio/Melisa/A.jpg"]
letras_melisa = ["M", "E", "L", "I", "S", "A"]

df_melisa = predecir_lote(rutas_melisa, letras_melisa, modelo, codificador, integrante="Melisa")
df_melisa

Aciertos (Melisa): 3/6 (50.0%)


,integrante,archivo,real,predicho,confianza,correcto
0,Melisa,./TestPropio/Melisa/M.jpg,M,W,1.0000,False
1,Melisa,./TestPropio/Melisa/E.jpg,E,nothing,0.9121,False
2,Melisa,./TestPropio/Melisa/L.jpg,L,L,0.9989,True
3,Melisa,./TestPropio/Melisa/I.jpg,I,I,0.9992,True
4,Melisa,./TestPropio/Melisa/S.jpg,S,S,1.0000,True
5,Melisa,./TestPropio/Melisa/A.jpg,A,Z,0.5396,False


In [10]:
rutas_renato = ["./TestPropio/Renato/R.jpg", "./TestPropio/Renato/E.jpg", "./TestPropio/Renato/N.jpg",
                "./TestPropio/Renato/A.jpg", "./TestPropio/Renato/T.jpg", "./TestPropio/Renato/O.jpg"]
letras_renato = ["R", "E", "N", "A", "T", "O"]

df_renato = predecir_lote(rutas_renato, letras_renato, modelo, codificador, integrante="Renato")
df_renato

Aciertos (Renato): 2/6 (33.3%)


,integrante,archivo,real,predicho,confianza,correcto
0,Renato,./TestPropio/Renato/R.jpg,R,R,0.9247,True
1,Renato,./TestPropio/Renato/E.jpg,E,S,0.8973,False
2,Renato,./TestPropio/Renato/N.jpg,N,N,0.9069,True
3,Renato,./TestPropio/Renato/A.jpg,A,T,0.7798,False
4,Renato,./TestPropio/Renato/T.jpg,T,Y,0.9721,False
5,Renato,./TestPropio/Renato/O.jpg,O,Y,0.7735,False


In [11]:
rutas_belen = ["./TestPropio/Belen/B.jpg", "./TestPropio/Belen/E_1.jpg", "./TestPropio/Belen/L.jpg",
                "./TestPropio/Belen/E_2.jpg", "./TestPropio/Belen/N.jpg"]
letras_belen = ["B", "E", "L", "E", "N"]

df_belen = predecir_lote(rutas_belen, letras_belen, modelo, codificador, integrante="Belen")
df_belen

Aciertos (Belen): 2/5 (40.0%)


,integrante,archivo,real,predicho,confianza,correcto
0,Belen,./TestPropio/Belen/B.jpg,B,B,0.9993,True
1,Belen,./TestPropio/Belen/E_1.jpg,E,S,0.5371,False
2,Belen,./TestPropio/Belen/L.jpg,L,L,0.9137,True
3,Belen,./TestPropio/Belen/E_2.jpg,E,S,0.9975,False
4,Belen,./TestPropio/Belen/N.jpg,N,T,0.5334,False


Completos:


In [12]:
df_todos = pd.concat([df_renato, df_melisa, df_belen], ignore_index=True)

print("Resumen general:")
print(df_todos.groupby("integrante")["correcto"].agg(["sum", "count"]))
print(f"\nAccuracy total en fotos propias: {df_todos['correcto'].mean():.1%}")

df_todos

Resumen general:
            sum  count
integrante            
Belen         2      5
Melisa        3      6
Renato        2      6

Accuracy total en fotos propias: 41.2%


,integrante,archivo,real,predicho,confianza,correcto
0,Renato,./TestPropio/Renato/R.jpg,R,R,0.9247,True
1,Renato,./TestPropio/Renato/E.jpg,E,S,0.8973,False
2,Renato,./TestPropio/Renato/N.jpg,N,N,0.9069,True
3,Renato,./TestPropio/Renato/A.jpg,A,T,0.7798,False
4,Renato,./TestPropio/Renato/T.jpg,T,Y,0.9721,False
5,Renato,./TestPropio/Renato/O.jpg,O,Y,0.7735,False
6,Melisa,./TestPropio/Melisa/M.jpg,M,W,1.0000,False
7,Melisa,./TestPropio/Melisa/E.jpg,E,nothing,0.9121,False
8,Melisa,./TestPropio/Melisa/L.jpg,L,L,0.9989,True
9,Melisa,./TestPropio/Melisa/I.jpg,I,I,0.9992,True
